# Coverage report (F08.6)

One row per configured city/dataset slice from the two validated
configuration files - config/city_registry.yml (which cities exist and
which are enabled) and config/project.yml (which datasets each city
registers). For every slice:

* `availability_state` - `disabled` (registry-owned), `unknown` (no
  ledger row yet), or the latest evidence-backed state from
  ops.source_file_availability (`available`, `not_published`,
  `invalid`).
* `load_status` - `pending` (DERIVED: no ops.ingestion_manifest row
  means the load never started; `pending` is never a stored status),
  or the latest stored status (`running`, `success`, `failed`).

Read-only: this report never writes tables. The environment widget
selects the test or live ops tables through the shared environment
resolver, exactly like every other ops notebook.

In [0]:
# Ops tables for the report, resolved through the shared environment
# resolver (F02.2): test -> the dedicated trippilot_test catalog,
# live -> trippilot_dev.ops.
dbutils.widgets.dropdown(
    "environment",
    "test",
    ["test", "live"],
    "Ops tables (test -> trippilot_test catalog)",
)

In [0]:
# Locate the repository root and import the control plane.
from pathlib import Path
import sys

start = Path.cwd().resolve()
ROOT = next(
    (
        p
        for p in [start, *start.parents]
        if (p / "src").is_dir() and (p / "config/project.yml").is_file()
    ),
    None,
)
if ROOT is None:
    raise RuntimeError(
        "Cannot locate repository root. Open this notebook inside the Databricks Git folder."
    )
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Repository root:", ROOT)

from src.common.control_plane import (
    build_city_dataset_coverage,
    load_city_registry,
    load_project_config,
)
from src.common.environment import resolve_table, validate_run_mode
from src.common.ingestion_manifest import IngestionManifestStore
from src.common.source_availability import AvailabilityStore

In [0]:
# Assemble the coverage grid from the registry, the project config and
# the environment's ops tables; read-only.
environment = validate_run_mode(dbutils.widgets.get("environment"))
registry = load_city_registry(ROOT)
project_config = load_project_config(ROOT)

availability_store = AvailabilityStore(
    table_name=resolve_table("source_availability", environment), spark=spark
)
ingestion_store = IngestionManifestStore(
    table_name=resolve_table("ingestion_manifest", environment), spark=spark
)

coverage_rows = build_city_dataset_coverage(
    project_config, registry, availability_store, ingestion_store
)

print(f"Coverage grid ({environment} mode): {len(coverage_rows)} city/dataset slices")
display(spark.createDataFrame(coverage_rows))